In [ ]:
!mkdir -p src/models scripts

In [ ]:
%%writefile src/models/two_head.py
"""DistilBERT encoder with two heads (SCHEMA §1.1, §1.2):
  * type head     : 8-way softmax (cross-entropy)
  * severity head : one value in 0-1 via sigmoid (MSE, masked for rows without severity)

Saved folder layout (models/classifier/):
  config.json + model.safetensors   <- encoder, Hugging Face save_pretrained format
  tokenizer files                    <- tokenizer.save_pretrained
  heads.pt                           <- type + severity head weights
  label_map.json                     <- SCHEMA §9, fixed order
"""
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

LABEL_MAP = {"FIRE": 0, "FLOOD": 1, "STORM_CYCLONE": 2, "EARTHQUAKE_COLLAPSE": 3,
             "TRANSPORT_ACCIDENT": 4, "MEDICAL": 5, "VIOLENCE_CRIME": 6, "NON_EMERGENCY": 7}
try:  # prefer the repo's single source of truth
    from src.data.labels import LABEL_MAP  # noqa: F811
except ImportError:
    pass
LABELS = sorted(LABEL_MAP, key=LABEL_MAP.get)


def sev_to_unit(s):
    """1-5 -> 0-1 (training scale)."""
    return (s - 1.0) / 4.0


def unit_to_sev(u):
    """0-1 -> 1-5 (output scale), clipped."""
    return (u * 4.0 + 1.0).clamp(1.0, 5.0) if torch.is_tensor(u) else min(5.0, max(1.0, u * 4.0 + 1.0))


class TwoHeadClassifier(nn.Module):
    def __init__(self, encoder, n_types=len(LABELS), dropout=0.1):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size if hasattr(encoder.config, "hidden_size") else encoder.config.dim
        self.dropout = nn.Dropout(dropout)
        self.type_head = nn.Linear(hidden, n_types)
        self.sev_head = nn.Linear(hidden, 1)
        self.n_types, self.p_drop = n_types, dropout

    @classmethod
    def from_encoder(cls, name_or_path, attn_implementation="sdpa", **kw):
        enc = AutoModel.from_pretrained(name_or_path, attn_implementation=attn_implementation)
        return cls(enc, **kw)

    def forward(self, input_ids, attention_mask, output_attentions=False):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_attentions=output_attentions)
        cls = self.dropout(out.last_hidden_state[:, 0])           # [CLS] vector
        type_logits = self.type_head(cls)
        sev_unit = torch.sigmoid(self.sev_head(cls)).squeeze(-1)  # 0-1
        return type_logits, sev_unit, (out.attentions if output_attentions else None)

    # ------------------------------------------------------------ save / load
    def save(self, folder, tokenizer=None):
        folder = Path(folder)
        folder.mkdir(parents=True, exist_ok=True)
        self.encoder.save_pretrained(folder)
        if tokenizer is not None:
            tokenizer.save_pretrained(folder)
        torch.save({"type_head": self.type_head.state_dict(), "sev_head": self.sev_head.state_dict(),
                    "n_types": self.n_types, "dropout": self.p_drop}, folder / "heads.pt")
        (folder / "label_map.json").write_text(json.dumps(LABEL_MAP, indent=2))

    @classmethod
    def load(cls, folder, device="cpu", attn_implementation="eager"):
        """eager attention so output_attentions works (needed for the rollout heatmap)."""
        folder = Path(folder)
        saved_map = json.loads((folder / "label_map.json").read_text())
        if saved_map != LABEL_MAP:
            raise ValueError(f"label_map.json in {folder} does not match LABEL_MAP")
        heads = torch.load(folder / "heads.pt", map_location=device)
        model = cls.from_encoder(folder, attn_implementation=attn_implementation,
                                 n_types=heads["n_types"], dropout=heads["dropout"])
        model.type_head.load_state_dict(heads["type_head"])
        model.sev_head.load_state_dict(heads["sev_head"])
        return model.to(device).eval()


def two_head_loss(type_logits, sev_unit, y_type, y_sev_unit, sev_weight=1.0):
    """Cross-entropy for type + masked MSE for severity.
    y_sev_unit is NaN where severity is missing; those rows add 0 and the MSE is divided by
    max(number of rows with severity, 1)."""
    ce = F.cross_entropy(type_logits, y_type)
    mask = ~torch.isnan(y_sev_unit)
    target = torch.where(mask, y_sev_unit, torch.zeros_like(y_sev_unit))
    sq = (sev_unit.float() - target.float()) ** 2 * mask.float()
    n = mask.sum()
    mse = sq.sum() / n.clamp(min=1)
    return ce + sev_weight * mse, ce.detach(), mse.detach(), int(n)


In [ ]:
%%writefile scripts/train_classifier.py
"""Train the DistilBERT two-head classifier (type + severity) and write prediction CSVs.

Kaggle (GPU, Internet on):
    python scripts/train_classifier.py --data /kaggle/input/<dataset folder> --out /kaggle/working/classifier_run

Writes:
    <out>/model/                        encoder + tokenizer + heads.pt + label_map.json
    <out>/predictions/<split>_predictions.csv   id, pred_type, p_<TYPE> x8, pred_severity
    <out>/train_metrics.json            per-epoch losses and val scores (for monitoring only)

Final metrics are computed by the lead from the prediction CSVs.
"""
import argparse
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

REPO_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))
from src.models.two_head import LABEL_MAP, LABELS, TwoHeadClassifier, sev_to_unit, two_head_loss, unit_to_sev  # noqa: E402

NON_EMERGENCY = "NON_EMERGENCY"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def load_csv(path, require_labels=True):
    df = pd.read_csv(path, dtype={"id": str}, keep_default_na=False)
    df["text"] = df["text"].astype(str)
    if "severity" in df.columns:
        df["severity"] = pd.to_numeric(df["severity"], errors="coerce")   # empty -> NaN
    else:
        df["severity"] = np.nan
    if require_labels or "incident_type" in df.columns:
        if "incident_type" in df.columns:
            bad = set(df["incident_type"]) - set(LABELS)
            if bad and require_labels:
                raise ValueError(f"{path}: labels not in LABEL_MAP: {sorted(bad)}")
            df["y"] = df["incident_type"].map(LABEL_MAP).fillna(-1).astype(int)
        elif require_labels:
            raise ValueError(f"{path}: no incident_type column")
    return df


class ReportDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.enc = tokenizer(df["text"].tolist(), truncation=True, max_length=max_len)
        self.y = df["y"].tolist() if "y" in df.columns else [-1] * len(df)
        sev = df["severity"].to_numpy(dtype=float)
        self.sev = [float(sev_to_unit(s)) if not math.isnan(s) else float("nan") for s in sev]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i],
                "y": self.y[i], "sev": self.sev[i]}


def make_collate(pad_id):
    def collate(items):
        T = max(len(x["input_ids"]) for x in items)
        ids = torch.full((len(items), T), pad_id, dtype=torch.long)
        mask = torch.zeros((len(items), T), dtype=torch.long)
        for i, x in enumerate(items):
            n = len(x["input_ids"])
            ids[i, :n] = torch.tensor(x["input_ids"])
            mask[i, :n] = 1
        return {"input_ids": ids, "attention_mask": mask,
                "y": torch.tensor([x["y"] for x in items]), "sev": torch.tensor([x["sev"] for x in items], dtype=torch.float)}
    return collate


@torch.no_grad()
def predict(model, loader, device, use_amp):
    model.eval()
    probs, sevs = [], []
    for b in loader:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits, sev_unit, _ = model(b["input_ids"].to(device), b["attention_mask"].to(device))
        probs.append(torch.softmax(logits.float(), -1).cpu())
        sevs.append(sev_unit.float().cpu())
    return torch.cat(probs).numpy(), torch.cat(sevs).numpy()


def macro_f1_present(y_true, y_pred):
    present = sorted(set(y_true))
    f1s = []
    for c in present:
        tp = int(((y_pred == c) & (y_true == c)).sum())
        fp = int(((y_pred == c) & (y_true != c)).sum())
        fn = int(((y_pred != c) & (y_true == c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * p * r / (p + r) if p + r else 0.0)
    return float(np.mean(f1s)) if f1s else 0.0


def write_predictions(df, probs, sev_unit, path):
    pred_idx = probs.argmax(1)
    pred_type = [LABELS[i] for i in pred_idx]
    sev = np.clip(sev_unit * 4.0 + 1.0, 1.0, 5.0)
    sev = np.where(np.array(pred_type) == NON_EMERGENCY, 1.0, sev)   # SCHEMA §1.2: NON_EMERGENCY is always 1.0
    out = pd.DataFrame({"id": df["id"], "pred_type": pred_type})
    for i, lab in enumerate(LABELS):
        out[f"p_{lab}"] = probs[:, i].round(6)
    out["pred_severity"] = sev.round(2)
    path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(path, index=False)
    print(f"wrote {path} ({len(out)} rows)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="/kaggle/input/resqpulse-data")
    ap.add_argument("--train_file", default="public_train.csv")
    ap.add_argument("--val_file", default="public_val.csv")
    ap.add_argument("--predict_files", nargs="*", default=["public_val.csv", "public_test.csv"],
                    help="CSV files (in --data) to write predictions for, e.g. public_val.csv public_test.csv test_synth.csv")
    ap.add_argument("--model_name", default="distilbert-base-uncased")
    ap.add_argument("--out", default="/kaggle/working/classifier_run")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--max_len", type=int, default=128)
    ap.add_argument("--warmup_ratio", type=float, default=0.06)
    ap.add_argument("--weight_decay", type=float, default=0.01)
    ap.add_argument("--sev_weight", type=float, default=1.0)
    ap.add_argument("--select", choices=["last", "best_val"], default="last",
                    help="which epoch to keep; default last because v0 val shares events with train")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--no_fp16", action="store_true")
    args = ap.parse_args()

    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda" and not args.no_fp16
    out = Path(args.out)
    data = Path(args.data)
    print(f"device={device} fp16={use_amp} model={args.model_name}")

    train_df = load_csv(data / args.train_file)
    val_df = load_csv(data / args.val_file)
    n_sev = int(train_df["severity"].notna().sum())
    print(f"train {len(train_df)} rows ({n_sev} with severity) | val {len(val_df)} rows")
    if n_sev == 0:
        print("NOTE: no training row has severity -> severity head receives no gradient (expected on v0)")

    tok = AutoTokenizer.from_pretrained(args.model_name)
    collate = make_collate(tok.pad_token_id)
    g = torch.Generator().manual_seed(args.seed)
    train_loader = DataLoader(ReportDataset(train_df, tok, args.max_len), batch_size=args.batch_size,
                              shuffle=True, collate_fn=collate, generator=g, num_workers=2)
    val_loader = DataLoader(ReportDataset(val_df, tok, args.max_len), batch_size=128, collate_fn=collate)

    model = TwoHeadClassifier.from_encoder(args.model_name).to(device)
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    params = [{"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": args.weight_decay},
              {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(params, lr=args.lr)
    total = args.epochs * len(train_loader)
    sched = get_linear_schedule_with_warmup(opt, int(args.warmup_ratio * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    history, best = [], -1.0
    model_dir = out / "model"
    for ep in range(1, args.epochs + 1):
        model.train()
        t0 = time.time()
        sums = {"loss": 0.0, "ce": 0.0, "mse": 0.0, "n_sev": 0}
        for step, b in enumerate(train_loader, 1):
            ids, mask = b["input_ids"].to(device), b["attention_mask"].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits, sev_unit, _ = model(ids, mask)
            loss, ce, mse, n = two_head_loss(logits.float(), sev_unit.float(), b["y"].to(device), b["sev"].to(device), args.sev_weight)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            sums["loss"] += loss.item(); sums["ce"] += ce.item(); sums["mse"] += mse.item(); sums["n_sev"] += n
            if step % 100 == 0:
                print(f"  epoch {ep} step {step}/{len(train_loader)} loss {sums['loss'] / step:.4f}", flush=True)
        steps = len(train_loader)

        probs, sev_unit = predict(model, val_loader, device, use_amp)
        y = val_df["y"].to_numpy()
        f1 = macro_f1_present(y, probs.argmax(1))
        acc = float((probs.argmax(1) == y).mean())
        vs = val_df["severity"].to_numpy()
        has = ~np.isnan(vs)
        val_sev_mae = float(np.abs((sev_unit[has] * 4 + 1) - vs[has]).mean()) if has.any() else None
        rec = {"epoch": ep, "train_loss": sums["loss"] / steps, "train_ce": sums["ce"] / steps,
               "train_sev_mse_unit": sums["mse"] / steps, "train_rows_with_severity": sums["n_sev"],
               "val_acc": acc, "val_macro_f1_present": f1, "val_sev_mae": val_sev_mae, "seconds": round(time.time() - t0)}
        history.append(rec)
        print(f"epoch {ep}: loss {rec['train_loss']:.4f} (ce {rec['train_ce']:.4f}, sev mse {rec['train_sev_mse_unit']:.4f}) | "
              f"val acc {acc:.4f} macro-F1 {f1:.4f} sev MAE {val_sev_mae} | {rec['seconds']}s", flush=True)

        if args.select == "last" or f1 > best:
            best = f1
            model.save(model_dir, tok)
            print(f"  saved model -> {model_dir}")

    (out / "train_metrics.json").write_text(json.dumps({"args": vars(args), "history": history}, indent=2))

    # ---------------- predictions with the saved model
    model = TwoHeadClassifier.load(model_dir, device=device, attn_implementation="sdpa")
    for fname in args.predict_files:
        path = data / fname
        if not path.exists():
            print(f"skip {fname}: not found in {data}")
            continue
        df = load_csv(path, require_labels=False)
        loader = DataLoader(ReportDataset(df, tok, args.max_len), batch_size=128, collate_fn=collate)
        probs, sev_unit = predict(model, loader, device, use_amp)
        write_predictions(df, probs, sev_unit, out / "predictions" / f"{Path(fname).stem}_predictions.csv")
    print("done")


if __name__ == "__main__":
    main()


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python scripts/train_classifier.py --data /kaggle/input/datasets/ggauravj05/resqpulse-data --out /kaggle/working/classifier_run --train_file train.csv --val_file val.csv --predict_files val.csv test_public.csv test_synth.csv gold_inputs.csv --epochs 3

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, f1_score
D = "/kaggle/input/datasets/ggauravj05/resqpulse-data"; P = "/kaggle/working/classifier_run/predictions"
for name in ["test_public", "test_synth"]:
    g = pd.read_csv(f"{D}/{name}.csv", dtype={"id": str}, keep_default_na=False)
    p = pd.read_csv(f"{P}/{name}_predictions.csv", dtype={"id": str})
    m = g.merge(p, on="id")
    print(f"\n=== {name} (n={len(m)}) macro-F1 {f1_score(m.incident_type, m.pred_type, average='macro'):.4f}")
    print(classification_report(m.incident_type, m.pred_type, digits=3, zero_division=0))
    sev = pd.to_numeric(m.severity, errors="coerce")
    if sev.notna().any():
        print(f"severity MAE {(m.pred_severity[sev.notna()] - sev[sev.notna()]).abs().mean():.3f}")

In [ ]:
%%writefile baseline_tfidf_lr.py
"""TF-IDF + Logistic Regression baseline for incident_type (SCHEMA §1.1).

Run in a Kaggle notebook (or locally):
    python scripts/baseline_tfidf_lr.py --data /kaggle/input/resqpulse-data --out outputs/baseline \
        --train_file train.csv --val_file val.csv --eval_files val.csv test_public.csv test_synth.csv

Prints macro precision / recall / F1 on val and test, and writes prediction CSVs in the
same format as the DistilBERT model (id, pred_type, p_<TYPE> x8, pred_severity).
pred_severity is left empty: the baseline does not predict severity.

Macro scores are reported two ways:
  * present  - averaged over classes that appear in that split's gold labels (default, fair on v0
               which has no MEDICAL rows)
  * all8     - averaged over all 8 classes (absent classes count as 0)
"""
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support

SEED = 42
REPO_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))
try:  # single source of truth when run inside the repo
    from src.data.labels import LABEL_MAP
except ImportError:  # identical to SCHEMA §9
    LABEL_MAP = {"FIRE": 0, "FLOOD": 1, "STORM_CYCLONE": 2, "EARTHQUAKE_COLLAPSE": 3,
                 "TRANSPORT_ACCIDENT": 4, "MEDICAL": 5, "VIOLENCE_CRIME": 6, "NON_EMERGENCY": 7}
LABELS = sorted(LABEL_MAP, key=LABEL_MAP.get)          # index order 0..7
IDX = list(range(len(LABELS)))

def load(data_dir, name):
    df = pd.read_csv(Path(data_dir) / name, dtype={"id": str}, keep_default_na=False)
    unknown = set(df["incident_type"]) - set(LABELS)
    if unknown:
        raise ValueError(f"{name}: labels not in LABEL_MAP: {sorted(unknown)}")
    df["y"] = df["incident_type"].map(LABEL_MAP)
    return df


def macro(y_true, y_pred, labels):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    return {"macro_precision": round(p, 4), "macro_recall": round(r, 4), "macro_f1": round(f, 4)}


def evaluate(name, y_true, y_pred):
    present = sorted(set(y_true))
    res = {"n": int(len(y_true)), "accuracy": round(accuracy_score(y_true, y_pred), 4),
           "present": macro(y_true, y_pred, present), "all8": macro(y_true, y_pred, IDX),
           "classes_present": [LABELS[i] for i in present]}
    print(f"\n=== {name} (n={res['n']}) ===")
    print(f"accuracy {res['accuracy']:.4f}")
    print("macro over classes present: P {macro_precision:.4f}  R {macro_recall:.4f}  F1 {macro_f1:.4f}".format(**res["present"]))
    print("macro over all 8 classes:   P {macro_precision:.4f}  R {macro_recall:.4f}  F1 {macro_f1:.4f}".format(**res["all8"]))
    missing = [l for i, l in enumerate(LABELS) if i not in present]
    if missing:
        print(f"(no gold rows for: {', '.join(missing)})")
    print(classification_report(y_true, y_pred, labels=present, target_names=[LABELS[i] for i in present],
                                digits=4, zero_division=0))
    return res


def write_predictions(df, probs, path):
    """probs: (n, 8) in LABEL_MAP order. Classes absent from training get probability 0."""
    out = pd.DataFrame({"id": df["id"], "pred_type": [LABELS[i] for i in probs.argmax(1)]})
    for i, lab in enumerate(LABELS):
        out[f"p_{lab}"] = probs[:, i].round(6)
    out["pred_severity"] = ""  # baseline has no severity head
    out.to_csv(path, index=False)
    print(f"wrote {path}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="/kaggle/input/resqpulse-data")
    ap.add_argument("--out", default="outputs/baseline")
    ap.add_argument("--train_file", default="train.csv")
    ap.add_argument("--val_file", default="val.csv", help="used only to pick C")
    ap.add_argument("--eval_files", nargs="*", default=["val.csv", "test_public.csv", "test_synth.csv"],
                    help="labelled files to score and write predictions for")
    args = ap.parse_args()
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    dfs = {"train": load(args.data, args.train_file), "val": load(args.data, args.val_file)}
    for name in args.eval_files:
        if (Path(args.data) / name).exists():
            dfs[Path(name).stem] = load(args.data, name)
        else:
            print(f"skip {name}: not found")
    for k, d in dfs.items():
        print(f"{k}: {len(d)} rows | " + ", ".join(f"{l}={c}" for l, c in d["incident_type"].value_counts().items()))

    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True, strip_accents="unicode")
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, lowercase=True)
    tr_text = dfs["train"]["text"]
    X = {"train": hstack([word.fit_transform(tr_text), char.fit_transform(tr_text)]).tocsr()}
    for k in dfs:
        if k == "train":
            continue
        X[k] = hstack([word.transform(dfs[k]["text"]), char.transform(dfs[k]["text"])]).tocsr()

    # choose C on val macro-F1 (classes present); test is never used for selection
    best = None
    for C in (0.5, 1.0, 2.0, 4.0, 8.0):
        clf = LogisticRegression(C=C, max_iter=3000, random_state=SEED)
        clf.fit(X["train"], dfs["train"]["y"])
        f1 = macro(dfs["val"]["y"], clf.predict(X["val"]), sorted(set(dfs["val"]["y"])))["macro_f1"]
        print(f"C={C:<4} val macro-F1 {f1:.4f}")
        if best is None or f1 > best[0]:
            best = (f1, C, clf)
    _, C, clf = best
    print(f"selected C={C}")

    results = {"model": "baseline_tfidf_lr", "C": C}
    for k in [Path(n).stem for n in args.eval_files if Path(n).stem in dfs]:
        p_trained = clf.predict_proba(X[k])                   # columns = clf.classes_ (classes seen in train)
        probs = np.zeros((p_trained.shape[0], len(LABELS)))
        probs[:, clf.classes_] = p_trained
        pred = probs.argmax(1)
        results[k] = evaluate(k, dfs[k]["y"].to_numpy(), pred)
        write_predictions(dfs[k], probs, out / f"baseline_{k}_predictions.csv")
    (out / "baseline_metrics.json").write_text(json.dumps(results, indent=2))
    print(f"wrote {out / 'baseline_metrics.json'}")


if __name__ == "__main__":
    main()


In [ ]:
!python baseline_tfidf_lr.py --data /kaggle/input/datasets/ggauravj05/resqpulse-data --out /kaggle/working/baseline_v4